## 0. 準備環境（直接執行，不必逐行看懂）
先依序執行下面兩格，安裝所需套件。環境已可用時會自動跳過。

**Colab 請分開執行。** 第一格安裝 Conda 後可能自動重啟；等重新連線，再執行第二格安裝 PyGMT 等套件。勿在安裝期間重複按執行。

此教材使用 PyGMT 0.17 / GMT 6.5。新的 Colab 執行環境仍需安裝。


In [ ]:
import importlib.util
import subprocess
import sys

# 在子程序檢查，避免安裝前先載入目前 kernel 的動態函式庫
check_code = """
import pygmt, pandas, numpy, ipywidgets, ipyleaflet, pyproj, xarray, netCDF4, scipy
import shutil
from pygmt.clib import Session
assert pygmt.__version__.lstrip('v').startswith('0.17.')
assert shutil.which('gs')
with Session() as session:
    assert session.info['version'].startswith('6.5.')
"""
try:
    ENV_READY = subprocess.run(
        [sys.executable, "-c", check_code], capture_output=True, timeout=30
    ).returncode == 0
except subprocess.TimeoutExpired:
    ENV_READY = False

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ENV_READY:
    print("環境已可用，跳過安裝。")
elif IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.13"])
    import condacolab
    condacolab.install()
else:
    print("本機模式：使用目前 Python 環境。")

In [ ]:
import importlib.util
import subprocess
import sys

# 在子程序檢查，避免安裝前先載入目前 kernel 的動態函式庫
check_code = """
import pygmt, pandas, numpy, ipywidgets, ipyleaflet, pyproj, xarray, netCDF4, scipy
import shutil
from pygmt.clib import Session
assert pygmt.__version__.lstrip('v').startswith('0.17.')
assert shutil.which('gs')
with Session() as session:
    assert session.info['version'].startswith('6.5.')
"""
try:
    ENV_READY = subprocess.run(
        [sys.executable, "-c", check_code], capture_output=True, timeout=30
    ).returncode == 0
except subprocess.TimeoutExpired:
    ENV_READY = False

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ENV_READY:
    print("環境已可用，跳過安裝。")
elif IN_COLAB:
    subprocess.check_call([
        "mamba", "install", "-y", "-c", "conda-forge",
        "pygmt=0.17", "gmt=6.5", "ghostscript=10.04", "pandas", "numpy", "ipywidgets", "ipyleaflet", "pyproj", "xarray", "netcdf4", "scipy"
    ])
else:
    print("跳過 Colab 安裝。")


# 03｜AI 探索與作業：畫世界的板塊交界帶

選一段世界的板塊交界帶，請 AI 用 PyGMT 畫出「海底地形＋地震分布」與「A–B 深度剖面」，再從圖上的證據說明這段交界屬於哪一類。

請先另存副本，完成環境設置後，由上往下執行。

[課前介紹](https://github.com/jimmy60504/pygmt-map-lab/blob/main/intro.md) · [課程首頁](https://github.com/jimmy60504/pygmt-map-lab)

[1｜基本地圖與地震](https://colab.research.google.com/github/jimmy60504/pygmt-map-lab/blob/main/01_maps_earthquakes.ipynb) · [2｜地形與 3D](https://colab.research.google.com/github/jimmy60504/pygmt-map-lab/blob/main/02_terrain_3d.ipynb) · [3｜AI 探索與作業](https://colab.research.google.com/github/jimmy60504/pygmt-map-lab/blob/main/03_ai_exploration.ipynb)

### 常用快捷鍵

| 操作 | Windows／Linux | Mac |
| --- | --- | --- |
| 執行目前儲存格並移到下一格 | Shift + Enter | Shift + Enter |
| 取消／切換註解 | Ctrl + / | ⌘ + / |

把游標放在程式行，或選取多行，再按切換註解的快捷鍵，即可移除或加上行首的 `#`。只選程式行，不要連中文說明一起取消註解。

修改後按 **Shift + Enter** 看結果，等執行完成再繼續下一步。


## 5. 用 AI 畫世界的板塊交界帶

前兩份 Notebook 用台灣練習了地圖、地震深度上色與地形。這一份把同一套方法搬到世界其他地方：**選一段板塊交界帶，畫出海底地形與地震分布，再切一條 A–B 剖面**，從圖上的證據說明它是哪一類交界。這堂課接著就是板塊構造，你畫的圖會是下堂課的討論材料。

事先知道答案沒關係，例如大家都知道日本是隱沒帶。重點是**你的圖能不能拿出證據**：海溝在哪裡、洋脊在哪裡、地震最深到幾公里、剖面上有沒有一條傾斜的震帶。判斷對不對不是主要分數，證據寫得清不清楚才是。

怎麼用 AI：

1. **先跑通課堂範本**，看懂參數在哪裡：範圍、時間、規模門檻、A–B 兩端、走廊半寬。
2. **把範本貼給 AI**，告訴它你要換哪個區域、剖面要往哪個方向切。程式能跑、圖出來，再檢查圖例、比例尺、深度分級與資料來源。
3. **自己先判讀**，再問 AI 意見。AI 很想直接講答案，可以先要求它不要說，等你寫完證據再對照。
4. **結果不符合也如實寫**：沒地震的地方可能是真的安靜，也可能是資料太少；分不出來就寫分不出來。

可以這樣起頭：

> 我在 Colab 用 PyGMT 0.17。這是課堂範本（貼上程式）。請把區域改成＿＿，地震改成 2000 年起 M ≥ 5，先用 USGS 的 count 查筆數，超過 20,000 就提高規模門檻。地圖用地形當底、地震依 0–70、70–300、300–700 km 三段上色，再畫一條大致垂直地震帶的 A–B 剖面，深度軸 0–700 km，標示 VE、比例尺與資料來源。先不要告訴我這裡是哪種板塊交界，我要自己判斷。

### 三大類交界帶，在地形與地震上長什麼樣

分類沿用 Lillie (1999)《Whole Earth Geophysics》第 2 章：張裂（divergent）、聚合（convergent）、轉形（transform）三大類，再各分小類。下表只講「圖上會看到什麼」，判讀時逐項對照：

| 大類 | 小類 | 地形訊號 | 地震訊號 | 剖面上的樣子 |
| --- | --- | --- | --- | --- |
| 聚合 | 海洋–海洋隱沒 | 深海溝，旁邊一串火山島弧，弧後常有海盆 | 由海溝往島弧方向逐漸變深，可達數百公里 | 一條傾斜的深震帶 |
| 聚合 | 海洋–大陸隱沒 | 海溝緊貼大陸邊緣，陸上有高山與火山鏈 | 同上，往大陸下方傾斜 | 一條傾斜的深震帶 |
| 聚合 | 大陸–大陸碰撞 | 沒有海溝，有極寬的高山與高原，火山少 | 淺到中深，分布寬而散 | 寬而淺的一片，沒有清楚的傾斜帶 |
| 張裂 | 中洋脊 | 海底長條隆起，慢速者中央有裂谷，被垂直的斷裂帶切成段 | 只有淺震，沿脊軸窄窄一條 | 全在最上層 |
| 張裂 | 大陸裂谷／年輕海洋 | 陸上長條裂谷、湖泊、火山；或狹長的新生海 | 淺震為主，沿裂谷分布 | 全在最上層 |
| 轉形 | 海洋／大陸轉形斷層 | 直線狀的地形錯動，無海溝也無火山鏈 | 淺震沿線排列，幾乎沒有中深震 | 全在最上層、窄 |

**深度分級**照常見慣例：淺 0–70 km、中 70–300 km、深 300–700 km。有中深震幾乎就是隱沒帶；只有淺震，就要靠地形分辨張裂或轉形。

三個提醒：

- **沒有地震不代表沒有交界**。有些隱沒帶幾十年安靜，目錄裡幾乎沒有點；地震少本身也是要寫進圖說的觀察。
- **深度不一定是量到的**。USGS 對洋脊與某些陸區的地震常直接填 10 km 或 33 km 的預設深度，畫出來會排成一直線。看到整排同深度，先查是不是預設值。
- **一個框可能同時有兩類**。交界帶會轉彎、分段；框裡看到兩種訊號是正常的，分段說明即可。

課本的板塊圖有版權，這裡用 USGS 公有領域的掛圖代替：[This Dynamic Planet（2006，正面）](https://pubs.usgs.gov/imap/2800/TDPfront-screen.pdf)，板塊、火山、地震與撞擊坑都在同一張圖上。

### 全球總覽：地震、火山與熱點

先看全球：地震帶就是交界帶，火山沿隱沒帶與裂谷排列，熱點大多在板塊內部。本格可獨立執行，會下載三份資料：

| 資料 | 來源 | 這次可以先看 |
| --- | --- | --- |
| 地震 | [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/) | `count` 端點先查筆數；單次查詢上限 20,000 筆 |
| 火山 | [NOAA NCEI 火山位置 API](https://www.ngdc.noaa.gov/hazel/view/hazards/volcano/loc-search)（源自 [Smithsonian GVP](https://volcano.si.edu/) 全新世清單） | 每頁最多 200 筆，需分頁 |
| 熱點 | GMT 範例檔 `@hotspots.txt`（Müller, Royer & Lawver 1993） | [pygmt.which()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.which.html) 自動下載 |
| 地形 | [GMT 全球地形](https://docs.generic-mapping-tools.org/latest/datasets/remote-data.html) `@earth_relief_10m` | 全球圖用粗解析度即可 |

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [pygmt.makecpt()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.makecpt.html) | 三段深度分級色票 | `series` 用逗號字串給不等距的分界 |
| [fig.grdimage()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdimage.html) | 全球地形底圖 | `region="d"`、`projection="N150/22c"` |
| [fig.plot()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.plot.html) | 地震、火山、熱點 | `style="c"`、`"t"`、`"a"` 分別是圓、三角、星形 |

In [ ]:
import json
import urllib.request
import pandas as pd
import pygmt

# 1. 全球地震：USGS 2000 年起，先查筆數，超過 20,000 就提高規模門檻
base = "https://earthquake.usgs.gov/fdsnws/event/1/"
for minmag in (5.5, 6.0, 6.5):
    query = f"starttime=2000-01-01&minmagnitude={minmag}"
    count = int(urllib.request.urlopen(base + "count?" + query).read())
    if count <= 20000:
        break
print(f"USGS 2000 年起 M >= {minmag}：{count} 筆")
quakes = pd.read_csv(base + "query?format=csv&" + query).dropna(subset=["longitude", "latitude", "depth", "mag"])
quakes = quakes.sort_values("depth", ascending=False)  # 深的先畫，淺的疊在上面

# 2. 火山：NOAA NCEI 全新世火山清單（源自 Smithsonian GVP），每頁 200 筆
rows, page = [], 1
while True:
    url = f"https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/volcanolocs?itemsPerPage=200&page={page}"
    with urllib.request.urlopen(url) as response:
        data = json.load(response)
    rows += data["items"]
    if page >= data["totalPages"]:
        break
    page += 1
volcanoes = pd.DataFrame(rows).dropna(subset=["latitude", "longitude"])
print(f"火山：{len(volcanoes)} 座")

# 3. 熱點：GMT 範例檔，第 1、2 欄是經度、緯度
hotspots = pd.read_csv(pygmt.which("@hotspots.txt", download="c"), sep=r"\s+",
                       comment="#", header=None, usecols=[0, 1], names=["lon", "lat"])
print(f"熱點：{len(hotspots)} 個")

# 4. 全球圖：地形底圖 + 地震三段深度 + 火山 + 熱點
fig = pygmt.Figure()
fig.grdimage(grid="@earth_relief_10m", region="d", projection="N150/22c", cmap="geo",
             shading="+a-45+nt0.4", frame=[f"+tGlobal seismicity (USGS M>={minmag} since 2000), volcanoes, hotspots", "g30"])
fig.coast(shorelines="0.15p,gray30")

pygmt.makecpt(cmap="#d7191c,#fdae61,#2c7bb6", series="0,70,300,700")  # 淺、中、深三段
fig.plot(x=quakes.longitude, y=quakes.latitude, style="c0.05c", fill=quakes.depth, cmap=True, transparency=40)
fig.plot(x=volcanoes.longitude, y=volcanoes.latitude, style="t0.1c", fill="white", pen="0.15p,black")
fig.plot(x=hotspots.lon, y=hotspots.lat, style="a0.3c", fill="yellow", pen="0.4p,black")
fig.colorbar(position="JBC+w8c/0.3c+h+o0c/0.8c", frame=["a0", "+lEarthquake depth (km): 0-70 / 70-300 / 300-700"])
fig.show()
print("白三角＝全新世火山；黃星＝熱點。找一條你想畫的地震帶，記下大概的經緯度。")

### 世界板塊交界帶整理：挑一段來畫

整理表另放在 [plate-boundaries.md](https://github.com/jimmy60504/pygmt-map-lab/blob/main/plate-boundaries.md)：隱沒帶、大陸碰撞、中洋脊、大陸裂谷、轉形斷層共三十多段，加一個「不是交界」的對照組，每段附範圍、兩側板塊與「佐證時注意」。從裡面挑一段，或自己框。

怎麼選：

1. 框 10–20 度寬，把海溝或洋脊整條框進去，交界兩側都留空間。
2. 先查筆數：2000 年起 M ≥ 5 多數區域在幾百到幾千筆；超過 20,000 提高規模，少於幾十筆就拉長時間或接受「這裡本來就少」。
3. A–B 大致垂直地震帶，半寬 50–150 km；混合型的框分段畫。
4. 建議配對：一段有中深震的、一段只有淺震的，對照最清楚。

### 區域範本：地圖＋A–B 剖面

只改最上面的參數即可換區域。本格獨立執行，會下載地震與地形，輸出兩張圖：地圖（地形、地震三段深度、A–B 線與走廊、位置示意）與剖面（上方地形、下方距離–深度）。地震深度用 USGS 目錄值，剖面是走廊內事件的投影。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/) | 查筆數與下載 | `count`、`minlongitude` 可用到 ±360 以跨換日線 |
| [pygmt.project()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.project.html) | 把走廊內地震投影到 A–B 線 | `center`、`endpoint`、`width`、`length="w"`、`generate` |
| [pygmt.grdtrack()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.grdtrack.html) | 沿 A–B 取樣地形 | `points`、`grid` |
| [fig.inset()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.inset.html) | 角落的地球儀位置圖 | `position`、`projection="G"` |
| [pyproj Geod](https://pyproj4.github.io/pyproj/stable/api/geod.html) | 算走廊邊界的球面座標 | `fwd`、`inv` |

In [ ]:
import urllib.request
import numpy as np
import pandas as pd
import pygmt
from pyproj import Geod

# ==== 改這裡：從整理表挑一段，或自己框 ====
REGION = [128, 150, 30, 46]        # 西、東、南、北
START, MINMAG = "2000-01-01", 5.0   # 地震起始日與最低規模
A, B = (130, 38.5), (148, 38.5)     # 剖面兩端（經度、緯度），大致垂直地震帶
HALF_WIDTH_KM = 100                 # 走廊半寬
DEPTH_MAX = 700                     # 剖面深度軸下限；固定 700 方便不同區域比較
# =========================================

# 1. 地震：先查筆數，超過 20,000 自動提高規模門檻
base = "https://earthquake.usgs.gov/fdsnws/event/1/"
minmag = MINMAG
while True:
    query = (f"starttime={START}&minmagnitude={minmag}"
             f"&minlongitude={REGION[0]}&maxlongitude={REGION[1]}&minlatitude={REGION[2]}&maxlatitude={REGION[3]}")
    count = int(urllib.request.urlopen(base + "count?" + query).read())
    if count <= 20000:
        break
    minmag += 0.5
quakes = pd.read_csv(base + "query?format=csv&" + query).dropna(subset=["longitude", "latitude", "depth", "mag"])
print(f"USGS {START} 起 M >= {minmag}：{len(quakes)} 筆")

# 2. 走廊內地震投影到 A–B 線：distance 沿線距離、offset 離線距離（km）
selected = pygmt.project(data=quakes[["longitude", "latitude", "depth", "mag"]],
                         center=list(A), endpoint=list(B), unit=True, length="w",
                         width=[-HALF_WIDTH_KM, HALF_WIDTH_KM], convention="xypqz")
selected.columns = ["longitude", "latitude", "distance", "offset", "depth", "mag"]
selected = selected.sort_values("depth", ascending=False)
track = pygmt.project(center=list(A), endpoint=list(B), generate=5, unit=True)
track.columns = ["lon", "lat", "distance"]
length_km = float(track.distance.iloc[-1])
print(f"走廊內 {len(selected)} 筆；A–B 長 {length_km:.0f} km")

# 3. 地形：地圖用 02m；剖面沿 A–B 取樣
grid = pygmt.datasets.load_earth_relief(resolution="02m", region=REGION)
topo = pygmt.grdtrack(points=track[["lon", "lat"]], grid=grid, newcolname="z")

# 4. 走廊外框：沿線每點往左右各推 HALF_WIDTH_KM（球面）
sphere = Geod(a=6371008.8, f=0)
azimuth, _, _ = sphere.inv(A[0], A[1], B[0], B[1])
left_lon, left_lat, _ = sphere.fwd(track.lon, track.lat, np.full(len(track), azimuth - 90), np.full(len(track), HALF_WIDTH_KM * 1000))
right_lon, right_lat, _ = sphere.fwd(track.lon, track.lat, np.full(len(track), azimuth + 90), np.full(len(track), HALF_WIDTH_KM * 1000))
corridor_lon = np.concatenate([left_lon, right_lon[::-1]])
corridor_lat = np.concatenate([left_lat, right_lat[::-1]])

def mag_size(m):
    return min(0.05 * 2 ** (m - 5), 0.4)  # 有上限，避免 M9 把整張圖蓋掉

# 5. 地圖
fig = pygmt.Figure()
fig.grdimage(grid=grid, region=REGION, projection="M12c", cmap="geo", shading="+a-45+nt0.5",
             frame=["af", f"+tUSGS M>={minmag} since {START[:4]}"])
fig.coast(shorelines="0.3p,gray20", resolution="i")
pygmt.makecpt(cmap="#d7191c,#fdae61,#2c7bb6", series="0,70,300,700")  # 淺、中、深
fig.plot(x=quakes.longitude, y=quakes.latitude, size=quakes.mag.apply(mag_size) * 0.7,
         fill=quakes.depth, cmap=True, style="c", pen="0.1p,black", transparency=40)
fig.plot(x=corridor_lon, y=corridor_lat, close=True, pen="0.8p,black,--")
fig.plot(x=[A[0], B[0]], y=[A[1], B[1]], pen="1.5p,black")
fig.text(x=[A[0], B[0]], y=[A[1], B[1]], text=["A", "B"], font="10p,Helvetica-Bold",
         fill="white", pen="0.5p,black", offset="0c/0.3c")
fig.basemap(map_scale=f"jBL+c{(REGION[2] + REGION[3]) / 2}+w500k+o0.3c/0.3c+f+lkm")
fig.colorbar(position="JBC+w8c/0.3c+h+o0c/0.8c", frame=["a0", "+lDepth (km): 0-70 / 70-300 / 300-700"])
with fig.inset(position="jTR+w2.5c+o0.1c"):
    fig.coast(region="g", projection=f"G{(REGION[0] + REGION[1]) / 2}/{(REGION[2] + REGION[3]) / 2}/2.5c",
              land="gray70", water="white", frame="g")
    fig.plot(x=[REGION[0], REGION[1], REGION[1], REGION[0]], y=[REGION[2], REGION[2], REGION[3], REGION[3]],
             close=True, pen="1p,red")
fig.show()

# 6. 剖面：上方地形（各自尺度），下方距離–深度
fig = pygmt.Figure()
fig.basemap(region=[0, length_km, -11, 9], projection="X14c/2c", frame=["Wsne", "ya5f1+lTopo (km)"])
fig.plot(x=track.distance, y=topo.z / 1000, pen="0.8p,black")
fig.plot(x=[0, length_km], y=[0, 0], pen="0.3p,gray50,--")
fig.text(text="A", position="TL", offset="0.15c/-0.1c", font="10p,Helvetica-Bold")
fig.text(text="B", position="TR", offset="-0.15c/-0.1c", font="10p,Helvetica-Bold")
fig.shift_origin(yshift="-8.3c")
fig.basemap(region=[0, length_km, 0, DEPTH_MAX], projection="X14c/-8c",
            frame=["WSne", "xa200f100+lDistance from A (km)", "ya100f50+lDepth (km)"])
pygmt.makecpt(cmap="#d7191c,#fdae61,#2c7bb6", series="0,70,300,700")  # 新的 Figure 要重建色票
fig.plot(x=selected.distance, y=selected.depth, size=selected.mag.apply(mag_size),
         fill=selected.depth, cmap=True, style="c", pen="0.3p,black", transparency=20)
ve = (8 / DEPTH_MAX) / (14 / length_km)
fig.text(text=f"corridor +/-{HALF_WIDTH_KM} km, {len(selected)} events, VE = {ve:.1f}x (topo panel separate)",
         position="BL", offset="0.15c/0.15c", font="8p", fill="white@30")
fig.show()

fixed = selected.depth.round(1).isin([10.0, 33.0, 35.0]).mean()
print(f"圖說骨架：USGS {START} 起 M >= {minmag}，範圍 {REGION}，A={A}、B={B}，走廊全寬 {2 * HALF_WIDTH_KM} km；"
      f"剖面 VE = {ve:.1f}x；走廊內 {fixed:.0%} 的深度是 USGS 預設值（10／33 km）。")

### 進階：把速度構造鋪在剖面底下

地震只標出正在破裂的位置；層析成像看得到冷的板片本身。這格沿用上一格的 `track`、`selected`、`length_km`，在同一條 A–B 底下鋪上 TX2019slab（Lu et al. 2019）的 P 波速度異常：藍色快，通常是冷的板片；紅色慢，通常是熱的地函。模型是全球 1°×1°、22 層，50 km 以下才有值，6.7 MB，首次執行會下載。

不想寫程式，也可以用 EarthScope 的線上工具直接出圖，再和自己的剖面並排：[EMC Cross-section Viewer](https://ds.iris.edu/dms/products/emc/emc-assets/about/generalized_cross_section_about.html)。

| 資料／指令 | 用途 | 這次可以先看 |
| --- | --- | --- |
| [EarthScope EMC](http://ds.iris.edu/ds/products/emc/)：[TX2019slab](https://data.earthscope.org/app/products/portal/emc_model_viewer.html?id=EMC-TX2019slab) | 全球 P 與 S 波層析模型 | 變數 `dvp`、`dvs`；引用 Lu, Grand, Lai & Garnero (2019) |
| [xarray.open_dataset()](https://docs.xarray.dev/en/stable/generated/xarray.open_dataset.html) | 讀 netCDF | `interp` 沿線內插 |
| [fig.grdimage()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.Figure.grdimage.html) | 畫距離–深度的二維格點 | 直接餵 DataArray |

In [ ]:
import os
import urllib.request
import numpy as np
import pygmt
import xarray as xr

# 先執行上一格，這裡沿用 track、selected、length_km、HALF_WIDTH_KM、DEPTH_MAX、mag_size

# 1. 下載模型（只下載一次）
MODEL = "TX2019slab.nc"
if not os.path.exists(MODEL):
    urllib.request.urlretrieve(
        "https://data.earthscope.org/archive/seismology/products/emc/netcdf/TX2019slab_percent.r0.0-n4c.nc", MODEL)
model = xr.open_dataset(MODEL)
VAR = "dvp"  # 或 "dvs"

# 2. 沿 A–B 內插：水平用模型格點雙線性內插，深度再內插成每 10 km
points = model[VAR].interp(longitude=("track", track.lon.values), latitude=("track", track.lat.values))
depths = np.arange(0, DEPTH_MAX + 10, 10)
slice_values = np.full((len(depths), len(track)), np.nan)
for j in range(len(track)):
    column = points.values[:, j]
    ok = np.isfinite(column)
    slice_values[:, j] = np.interp(depths, points.depth.values[ok], column[ok], left=np.nan, right=np.nan)
tomo = xr.DataArray(slice_values, coords={"y": depths, "x": track.distance.values}, dims=("y", "x"))

# 3. 速度剖面 + 地震
fig = pygmt.Figure()
pygmt.makecpt(cmap="roma", series=[-2, 2, 0.1])  # 紅慢、藍快
fig.grdimage(grid=tomo, region=[0, length_km, 0, DEPTH_MAX], projection="X14c/-8c", cmap=True, nan_transparent=True,
             frame=[f"WSne+t{VAR} TX2019slab (Lu et al. 2019)", "xa200f100+lDistance from A (km)", "ya100f50+lDepth (km)"])
fig.colorbar(position="JBC+w8c/0.3c+h+o0c/1c", frame=["a1", f"+l{VAR} (%): red slow, blue fast"])
pygmt.makecpt(cmap="#d7191c,#fdae61,#2c7bb6", series="0,70,300,700")
fig.plot(x=selected.distance, y=selected.depth, size=selected.mag.apply(mag_size),
         fill=selected.depth, cmap=True, style="c", pen="0.3p,black")
fig.text(text="A", position="TL", offset="0.15c/-0.1c", font="10p,Helvetica-Bold")
fig.text(text="B", position="TR", offset="-0.15c/-0.1c", font="10p,Helvetica-Bold")
fig.text(text=f"corridor +/-{HALF_WIDTH_KM} km; model 1x1 deg, no data above 50 km",
         position="BL", offset="0.15c/0.15c", font="8p", fill="white@30")
fig.show()
print("讀圖：藍色帶是否傾斜？地震是否貼著它？碰撞帶的高速體是平的，隱沒帶的是斜的。全球模型解析度約 1 度，細節看不到。")

### 選一個 AI 工具，開始做圖

課堂以 **Codex** 示範；你也可以用自己熟悉的工具，不必全部安裝，也不用為這堂課特別付費訂閱。重點是把問題說清楚、跑出圖，再檢查結果。

| 工具與官方資源 | 這堂課可以怎麼用 | 帳號與注意事項 |
| --- | --- | --- |
| [Codex](https://developers.openai.com/codex/auth/)（主要示範） | 使用前堂安裝的工具，協助修改程式、排除錯誤與加入互動功能。 | 有可使用 Codex 的 ChatGPT 帳號／訂閱，就用自己的帳號登入；額度以帳號顯示為準。 |
| [Claude Code](https://code.claude.com/docs/en/quickstart) | 已習慣 Claude 的同學，可以用它完成同樣的作圖任務。 | 已有支援 Claude Code 的訂閱（例如 Pro／Max）可使用自己的帳號，仍有使用額度限制。 |
| [Colab Gemini](https://research.google.com/colaboratory/faq.html) | 直接在 Notebook 的 Gemini 面板討論、修改程式或請它協助看錯誤，不必另外安裝 agent。 | 使用自己的 Google 帳號；功能是否開放受帳號資格、地區與額度限制，以介面為準。 |
| [OpenCode](https://opencode.ai/docs/)／[免費模型資訊](https://opencode.ai/docs/zen/) | 另一個 coding agent 選擇；可以先用當下提供的免費模型，從修改範例與簡單作圖開始。 | OpenCode 是工具，背後可選不同模型；請確認模型標示為免費。免費名單與供應狀況會變動，不代表所有模型都免費。 |

**怎麼選？** 跟著課堂就用 Codex；已有 Claude Code 或 Codex 訂閱就沿用；不想另裝工具可先用 Colab Gemini，想試其他 agent 則可用 OpenCode 的免費模型。遇到額度限制可以換工具，不必急著付費。

使用外部 agent 時，要提供目前的程式與完整錯誤訊息，並告訴它執行環境是 Colab、使用 PyGMT。電腦裡修改的檔案不會自動同步到 Colab，更新後仍要在 Colab 執行確認。

訂閱登入和付費 API 是不同的使用方式；本課不要求購買 API 額度。不要把 API key、密碼或登入憑證放進 Notebook／GitHub，使用電腦教室公用電腦後記得登出。


### AI 畫完後，檢查這張圖說清楚了嗎？

程式能跑、圖看起來漂亮，不代表讀者就能正確理解。把圖和圖說放在一起，檢查下面幾件事：

| 檢查項目 | 要注意什麼 |
| --- | --- |
| **主題與圖說** | 一眼能看出你想表達什麼嗎？圖說交代資料與範圍、如何呈現、實際觀察到什麼；分清楚觀測、模型與推論，不把原本的猜想直接寫成結論。 |
| **座標與單位** | 軸的名稱、單位、時間基準／時區是否清楚？是線性還是對數尺度？地圖用經緯網定位、用經緯網或北箭頭辨向；剖面兩端標出 A、B 與方位。 |
| **地圖比例尺** | 加上標有距離單位的比例尺，讓讀者判斷空間尺度；更換投影或範圍時要重新確認，局部放大圖也要有自己的尺度資訊。 |
| **圖例與色條** | 點、線、符號、大小、顏色各代表什麼？數值色條要有單位；超出色階與缺資料怎麼表示？規模不等於震度。 |
| **尺度與子圖對照** | 多張圖的顏色、大小與座標尺度能直接比較嗎？A–B 方向、子圖編號與框選範圍是否一致？若尺度不同，要明確交代。 |
| **篩選與不確定性** | 哪些時間、區域或規模被保留？是否做過平滑、插值或正規化？空白不一定代表沒有現象，可能只是缺資料；需要時呈現誤差或模型限制。 |
| **可讀性** | 縮到實際觀看大小後，字、線和圖例還看得清楚嗎？大圓、標籤、等高線或陰影有沒有遮住重點？ |

**剖面拉伸：標出垂直誇大倍率（VE）**

淺部剖面可以拉伸垂直方向，讓起伏更清楚，但要標示例如 `VE = 5×`。先把水平與垂直的實際距離換成同一單位，再比較它們在圖上的長度比例；相同實際距離若垂直畫得比水平長五倍，就是五倍誇大。不能只拿圖框的高／寬當成 VE，也不能直接用拉伸後的圖量斷層傾角。旋轉的 3D 透視圖則另有視角影響，不要把 `zsize` 直接當成 VE。

**局部放大：讓讀者知道這一小塊在哪裡**

局部地圖可在角落加一張較大範圍的位置示意圖（inset），用小框標出主圖涵蓋的範圍；也可以在大範圍主圖框出一區，再用另一張圖放大細節。框線、連接線或 `(a)、(b)` 要能清楚對照，並注意放大前後是否使用不同尺度。縮圖以定位為主，不必堆滿資料。

**顏色：不要只靠「看起來漂亮」來選**

- 重要類別不要只靠紅、綠區分，也可搭配符號、線型或文字。
- 有大小次序的量、正負異常與不同類別，適合的色票不一樣；顏色變化應幫助讀者理解資料。
- 用色覺模擬檢查；灰階可輔助檢查，但不能取代色覺測試。社群常用的色票也不一定是最友善的選擇。

可參考 [Seismica 的色票要求](https://seismica.library.mcgill.ca/submission-checklist) 與 [Nature 圖像規範](https://research-figure-guide.nature.com/figures/preparing-figures-our-specifications/)。各期刊要求不同，投稿時再確認目標期刊的規定。

**這份作業另外檢查三項**

- 深度分級的界線（70、300 km）有沒有寫在圖例上？
- 剖面線是否大致垂直地震帶？走廊半寬多少、幾筆事件，有沒有寫？
- 圖說有沒有寫下載日期、規模門檻，以及多少深度是 USGS 預設值？

可以請 AI 協助逐項檢查，但座標、單位、VE、資料來源和圖說，仍要自己對照資料確認。


### 先逛逛論文的圖，找找靈感

先看 [地震學常見圖像](https://github.com/jimmy60504/pygmt-map-lab/blob/main/earthquake-figure-guide.md) 認識各種圖型與範例，再看 [論文圖收集](https://github.com/jimmy60504/pygmt-map-lab/blob/main/figure-examples.md) 比較漂亮的和普通的。

不用一開始就讀懂整篇 paper。先到 [Google Scholar](https://scholar.google.com/) 搜尋 `seismic`、`earthquake` 或 `seismicity`，也可以加上 `Taiwan`、`subduction`、`cross section`、`waveform` 等地區或圖像關鍵字。或先用 Google 圖片搜尋，看到有興趣的圖，再回到原論文看圖說。

也可以直接逛這些期刊，挑一篇題目有興趣的文章，先翻圖片：

| 期刊入口 | 主要範圍 | 逛圖時可以找什麼 |
| --- | --- | --- |
| [SRL — Seismological Research Letters](https://pubs.geoscienceworld.org/srl) | 地震學及相關觀測、方法與應用 | 地震事件、測站、波形與資料展示。 |
| [BSSA — Bulletin of the Seismological Society of America](https://pubs.geoscienceworld.org/bssa) | 地震學與相關研究 | 地震分布、震源、地動與分析結果。 |
| [GJI — Geophysical Journal International](https://academic.oup.com/gji) | 固體地球物理，不限地震 | 地下構造、剖面、波形與模型比較。 |
| [GRL — Geophysical Research Letters](https://agupubs.onlinelibrary.wiley.com/journal/19448007) | 地球與太空科學，不限地震 | 搜尋地震相關文章，看作者怎麼用少量圖呈現重點。 |
| [Seismica](https://seismica.library.mcgill.ca/) | 地震學與地震科學，開放取用 | 地震研究、資料與方法的各種呈現方式。 |

挑一張喜歡的圖就好，想想：**它想表達什麼？資料怎麼篩選或排列？我可以借用哪種畫法來表達自己的問題？** 重點是學呈現方式，不是照抄結論，也不必做出同樣複雜的研究。遇到付費文章，可找開放版本或換一篇。

把原論文連結與圖號留給自己，也可以給 AI 當討論參考。圖片搜尋只是入口，仍要回原文確認圖說；若要把原圖放進公開 GitHub，需確認授權並標明來源。



### 這份作業會用到的資料

課堂範本已經把前四項接好；後面幾項是佐證用的補充，作品仍需用到 PyGMT，可搭配其他工具。

| 想找什麼 | 資料入口 | 可以做什麼 |
| --- | --- | --- |
| 全球地震目錄 | [USGS](https://earthquake.usgs.gov/fdsnws/event/1/)／[ISC Bulletin](https://www.isc.ac.uk/iscbulletin/search/) | 位置、深度、規模；USGS 單次上限 20,000 筆，ISC 整合各國網、小地震較全 |
| 台灣更細的地震 | [氣象署 GDMS](https://gdms.cwa.gov.tw/) | 想把台灣當對照組時用 |
| 海陸地形 | [GMT 全球地形](https://docs.generic-mapping-tools.org/latest/datasets/remote-data.html)／[GEBCO](https://www.gebco.net/data-products/gridded-bathymetry-data) | 海溝、洋脊、裂谷、斷裂帶；大框用 05m，細看用 01m 或 15s |
| 火山 | [NOAA NCEI 火山位置](https://www.ngdc.noaa.gov/hazel/view/hazards/volcano/loc-search)／[Smithsonian GVP](https://volcano.si.edu/) | 火山鏈平行海溝是隱沒帶、沿裂谷是張裂、轉形帶沒有 |
| 熱點 | GMT `@hotspots.txt`（Müller et al. 1993） | 板塊內部的火山，當「不是交界」的對照 |
| 板塊邊界線 | [Bird (2003) PB2002](http://peterbird.name/publications/2003_pb2002/2003_pb2002.htm)（[GeoJSON](https://github.com/fraxen/tectonicplates)）／[Hasterok et al. (2022)](https://github.com/dhasterok/global_tectonics) | 判讀完再疊上去對答案；PB2002 每段有類型碼 |
| 板片深度 | [Slab2（USGS）](https://www.sciencebase.gov/catalog/item/5aa1b00ee4b0b1c392e86467) | 隱沒帶的板片幾何，可疊在剖面上檢查傾斜帶 |
| 速度構造 | [EarthScope EMC](http://ds.iris.edu/ds/products/emc/) | 層析模型的剖面，看板片與岩石圈 |
| 震源機制 | [Global CMT](https://www.globalcmt.org/CMTfiles.html) | 逆衝、正斷層、走滑各對應聚合、張裂、轉形；用 `fig.meca()` 畫 |

下載前請 AI 一起檢查年份、座標系統、單位與授權。深度、規模的定義各目錄不同，不要混用；資料疊在一起不代表已證明因果。

### 選用：在地圖上點 A、B 切剖面

想用滑鼠決定剖面位置，可以用這格的互動地圖；改最上面的 `REGION`、`START`、`MINMAG` 就能換區域。完成環境安裝後本格可獨立執行。

1. 在互動地圖點一下設定 **A**，再點一下設定 **B**。
2. 用「半寬 km」拉桿決定剖面線兩側要選多寬；藍色範圍就是取樣走廊。
3. 按「更新剖面」，先看標有 A、B、剖面線與取樣走廊的 PyGMT 地圖，再看上下對齊的地形與地震剖面。換位置前按「重選 A、B」。

**讀圖注意**：深度向下增加；超過 `DEPTH_MAX` 的事件沿用色條最深端顏色。地形沿 A–B 中心線取樣，地震取線兩側走廊內的事件；兩個面板各自標示 VE，不能直接量圖上的傾角。已下載的資料只涵蓋 `REGION`，在框外畫剖面不會自動取得新地震。

互動地圖需網路與運作中的 Colab／Jupyter；GitHub 靜態預覽無法點選。若 Colab 提示允許自訂元件，請確認後啟用。

| 指令／官方 API | 用途 | 這次可以先看 |
| --- | --- | --- |
| [ipyleaflet Map](https://ipyleaflet.readthedocs.io/en/latest/map_and_basemaps/map.html) | 互動地圖與點擊事件 | `on_interaction`、`coordinates` |
| [pygmt.project()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.project.html) | 投影及篩選剖面地震 | `center`、`endpoint`、`width`、`length`、`unit` |
| [Jupyter Widgets](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html) | 拉桿與按鈕 | `IntSlider`、`Button`、`Output` |
| [pygmt.grdtrack()](https://www.pygmt.org/v0.17.0/api/generated/pygmt.grdtrack.html) | 沿 A–B 取樣地形 | `points`、`grid`、`newcolname` |

In [ ]:
import pygmt
from io import StringIO
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipyleaflet import Map, Marker, Polyline, Polygon, CircleMarker, LayerGroup
from IPython.display import display, clear_output
from datetime import datetime, timezone
from pyproj import Geod

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass  # 本機 Jupyter 不需要 Colab 的元件管理器


# ==== 改這裡 ====
REGION = [128, 150, 30, 46]        # 西、東、南、北
START, MINMAG = "2000-01-01", 5.0   # 地震起始日與最低規模
DEPTH_MAX = 700                     # 色條與剖面的深度上限
# =================

# 1. 本格自行下載資料，不依賴其他 cell
endtime = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
    f"&starttime={START}"
    f"&endtime={endtime}"
    f"&minmagnitude={MINMAG}"
    f"&minlongitude={REGION[0]}&maxlongitude={REGION[1]}"
    f"&minlatitude={REGION[2]}&maxlatitude={REGION[3]}"
)
quakes = pd.read_csv(url).dropna(subset=["longitude", "latitude", "depth", "mag"])

def magnitude_size(magnitude):
    if magnitude < 3:
        return 0.035
    elif magnitude < 4:
        return 0.07
    elif magnitude < 5:
        return 0.14
    elif magnitude < 6:
        return 0.28
    elif magnitude < 7:
        return 0.56
    else:
        return 1.12



def size_legend(map_symbols=False):
    rows = ""
    if map_symbols:
        rows += "S 0.6c c 0.1c gray70 - 1.4c Catalog events\n"
        rows += "S 0.6c - 0.8c - 1p,blue,-- 1.4c Sampling corridor\n"
        rows += "S 0.6c - 0.8c - 1.5p,black 1.4c A-B centerline\n"
        rows += "S 0.6c s 0.22c white 1p,black 1.4c A/B endpoint\n"
        rows += "G 0.2c\n"
    rows += "H 10p,Helvetica-Bold Magnitude\n"
    for mag, label in [(2,"M < 3"),(3,"3 <= M < 4"),(4,"4 <= M < 5"),
                       (5,"5 <= M < 6"),(6,"6 <= M < 7"),(7,"M >= 7")]:
        rows += f"S 0.6c c {magnitude_size(mag):.3f}c gray70 0.2p,gray30 1.4c {label}\n"
        rows += f"G {max(0.15,magnitude_size(mag)-0.25):.2f}c\n"
    return StringIO(rows)

# 2. 地圖只負責點選；GMT project 計算沿線距離與垂直距離
# 使用球面近似，走廊邊界也用同一種球面幾何
sphere = Geod(a=6371008.8, f=0)
points = []  # 點擊順序：A、B；每個座標為（緯度、經度）
map_view = Map(center=((REGION[2] + REGION[3]) / 2, (REGION[0] + REGION[1]) / 2),
               zoom=7 if REGION[1] - REGION[0] <= 6 else 4, scroll_wheel_zoom=True,
               layout=widgets.Layout(height="450px"))
earthquake_layer = LayerGroup(layers=tuple(
    CircleMarker(location=(row.latitude, row.longitude), radius=2,
                 color="#777777", fill_color="#777777", fill_opacity=0.4, weight=0)
    for row in quakes.itertuples()
))
selection_layer = LayerGroup()
map_view.add(earthquake_layer)
map_view.add(selection_layer)

width_slider = widgets.IntSlider(value=100, min=10, max=200, step=10,
    description="半寬 km", continuous_update=False)
update_button = widgets.Button(description="更新剖面", button_style="primary")
reset_button = widgets.Button(description="重選 A、B")
status = widgets.HTML("請在地圖點 A，再點 B。灰點為已下載的地震。")
output = widgets.Output()


def profile_geometry():
    (lat_a, lon_a), (lat_b, lon_b) = points
    azimuth, _, meters = sphere.inv(lon_a, lat_a, lon_b, lat_b)
    if meters < 1000:
        raise ValueError("A、B 請至少相距 1 km。")
    along = np.linspace(0, meters, 201)
    lons, lats, back = sphere.fwd(
        np.full(201, lon_a), np.full(201, lat_a), np.full(201, azimuth), along
    )
    # 軌跡每個位置的前進方向，左右各延伸半寬
    left_lon, left_lat, _ = sphere.fwd(lons, lats, back + 90, np.full(201, width_slider.value * 1000))
    right_lon, right_lat, _ = sphere.fwd(lons, lats, back - 90, np.full(201, width_slider.value * 1000))
    track = list(zip(lats, lons))
    corridor = list(zip(left_lat, left_lon)) + list(zip(right_lat, right_lon))[::-1]
    return meters / 1000, track, corridor


# 3. 點選或改寬度時，更新地圖範圍並清掉舊剖面
def refresh_selection():
    with output:
        clear_output(wait=False)
    layers = [Marker(location=p, title=label, draggable=False) for p, label in zip(points, ["A", "B"])]
    if len(points) == 2:
        try:
            length, track, corridor = profile_geometry()
        except ValueError as error:
            status.value = str(error)
        else:
            layers += [Polygon(locations=corridor, color="#1565c0", fill_opacity=0.12),
                       Polyline(locations=track, color="#1565c0", weight=3)]
            status.value = f"A={points[0]}；B={points[1]}；長約 {length:.0f} km，全寬 {2*width_slider.value} km。請按更新剖面。"
    else:
        status.value = "請點 B。" if points else "請點 A，再點 B。"
    selection_layer.layers = tuple(layers)

def on_map_click(**event):
    if event.get("type") != "click":
        return
    if len(points) == 2:
        status.value = "要換位置，請先按「重選 A、B」。"
        return
    points.append(tuple(event["coordinates"]))
    refresh_selection()

def reset_selection(_):
    points.clear()
    refresh_selection()


# 4. 按鈕才觸發計算與 GMT 出圖；深度向下增加
def draw_section(_=None):
    with output:
        clear_output(wait=True)
        if len(points) != 2:
            print("請先在地圖選好 A、B。")
            return
        try:
            length, track, corridor = profile_geometry()
        except ValueError as error:
            print(error)
            return

        selected = pygmt.project(
            data=quakes[["longitude", "latitude", "depth", "mag"]],
            center=list(points[0][::-1]),  # GMT 順序：經度、緯度
            endpoint=list(points[1][::-1]),
            unit=True,  # 距離單位 km
            length="w",  # 只保留 A 到 B 之間
            width=[-width_slider.value, width_slider.value],  # 線兩側的半寬
            convention="xypqz",  # 經度、緯度、沿線距離、離線距離、深度、規模
        )
        if selected.empty:
            print("範圍內沒有地震，請改位置或加大半寬。")
            return
        selected.columns = ["longitude", "latitude", "distance", "offset", "depth", "mag"]
        selected = selected.sort_values("mag")
        print(f"取樣 {len(selected)} / {len(quakes)} 筆；{START} 至 {endtime} UTC")
        print(f"只含下載範圍 {REGION}；走廊超出此範圍的部分沒有資料。")

        # 新增：先畫地理位置圖，灰點為全部地震，彩色點為剖面取樣
        map_fig = pygmt.Figure()
        pygmt.makecpt(cmap="batlow", series=[0, DEPTH_MAX, 1], continuous=True, background=True)
        corridor_lat, corridor_lon = np.array(corridor).T
        track_lat, track_lon = np.array(track).T
        map_fig.coast(
            region=[min(REGION[0], corridor_lon.min()-0.1), max(REGION[1], corridor_lon.max()+0.1),
                    min(REGION[2], corridor_lat.min()-0.1), max(REGION[3], corridor_lat.max()+0.1)],
            projection="M14c",
            land="gray95", water="aliceblue", shorelines="0.5p,gray40",
            frame=["af", "+tA-B profile location"],
        )
        map_fig.plot(x=quakes.longitude, y=quakes.latitude, style="c0.035c", fill="gray70")
        map_fig.plot(
            x=selected.longitude, y=selected.latitude,
            style="c", size=selected.mag.apply(magnitude_size),
            fill=selected.depth, cmap=True, transparency=40, pen="0.2p,gray30",
        )

        # 走廊只畫外框，避免蓋住地震的深度顏色
        map_fig.plot(x=corridor_lon, y=corridor_lat, close=True, pen="1p,blue,--")
        map_fig.plot(x=track_lon, y=track_lat, pen="1.5p,black")
        map_fig.plot(x=[points[0][1], points[1][1]], y=[points[0][0], points[1][0]],
                     style="s0.22c", fill="white", pen="1p,black")
        map_fig.text(
            x=[points[0][1], points[1][1]], y=[points[0][0], points[1][0]],
            text=["A", "B"], font="14p,Helvetica-Bold,black",
            offset="0.2c/0.2c", justify="BL", fill="white",
        )
        map_fig.basemap(map_scale=f"jBR+c{(points[0][0]+points[1][0])/2}+w{50 if REGION[1]-REGION[0] <= 6 else 500}k+o0.5c/0.5c+f+lkm")
        map_fig.colorbar(position="JBC+w10c/0.35c+h+e", frame=["xaf", "y+lDepth (km)"])
        map_fig.legend(spec=size_legend(map_symbols=True), position="JMR+jML+o0.5c+w4.5c", box="+gwhite+p0.5p")
        map_fig.show()
        print("地圖圖例：灰點＝下載目錄；彩色點＝剖面取樣事件；藍色虛線＝取樣走廊；黑線＝A–B 中心線；方形＝端點。")

        # 地形沿中心線取樣；地震使用線兩側的取樣走廊
        relief = pygmt.datasets.load_earth_relief(
            resolution="02m",
            region=[track_lon.min()-0.05, track_lon.max()+0.05,
                    track_lat.min()-0.05, track_lat.max()+0.05],
            registration="gridline",
        )
        terrain = pygmt.grdtrack(
            points=pd.DataFrame({"longitude": track_lon, "latitude": track_lat}),
            grid=relief,
            newcolname="elevation",
        )
        distance = np.linspace(0, length, len(terrain))
        height_km = terrain.elevation.to_numpy() / 1000
        if not np.isfinite(height_km).all():
            print("地形取樣有缺值，請調整 A、B；不以插值補成完整地形。")
            return

        # 同一水平距離軸，兩個面板各自計算垂直誇大倍率
        topo_min = min(-0.5, np.floor(height_km.min()*2)/2 - 0.5)
        topo_max = max(0.5, np.ceil(height_km.max()*2)/2 + 0.5)
        depth_min = min(0, np.floor(selected.depth.min()/50)*50)
        depth_max = max(50, np.ceil(selected.depth.max()/50)*50)
        topo_ve = (3 / (topo_max-topo_min)) / (14 / length)
        quake_ve = (9 / (depth_max-depth_min)) / (14 / length)
        # 依兩端的向外方位標示左、右方向（八方位）
        forward, backward, _ = sphere.inv(points[0][1], points[0][0], points[1][1], points[1][0])
        compass = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
        left_direction = compass[int(((forward + 180) % 360 + 22.5) // 45) % 8]
        right_direction = compass[int(((backward + 180) % 360 + 22.5) // 45) % 8]

        fig = pygmt.Figure()
        pygmt.makecpt(cmap="batlow", series=[0, DEPTH_MAX, 1], continuous=True, background=True)
        fig.basemap(
            region=[0, length, depth_min, depth_max],
            projection="X14c/-9c",
            frame=["xaf+lDistance from A (km)", "yaf+lDepth (km)", "WSen"],
        )
        fig.plot(x=selected.distance, y=selected.depth, style="c",
                 size=selected.mag.apply(magnitude_size), fill=selected.depth,
                 cmap=True, transparency=40, pen="0.2p,gray30")
        fig.text(position="TR", text=f"Earthquakes | VE = {quake_ve:.2f}x",
                 justify="TR", offset="-0.15c/-0.15c", font="10p", fill="white")
        fig.colorbar(position="JBC+w10c/0.35c+h+e", frame=["xaf", "y+lDepth (km)"])
        fig.legend(spec=size_legend(), position="JMR+jML+o0.5c+w4c", box="+gwhite+p0.5p")

        # 上方地形面板；0 km 虛線為海平面
        fig.shift_origin(yshift="10c")
        fig.basemap(
            region=[0, length, topo_min, topo_max],
            projection="X14c/3c",
            frame=["xf", "yaf+lElevation (km)", "WSen"],
        )
        fig.plot(x=distance, y=height_km, pen="1p,black")
        fig.plot(x=[0,length], y=[0,0], pen="0.6p,blue,--")
        fig.text(position="TL", text=f"A ({left_direction})    Terrain | VE = {topo_ve:.2f}x",
                 justify="TL", offset="0.15c/-0.15c", font="10p", fill="white")
        fig.text(position="TR", text=f"B ({right_direction})", justify="TR", offset="-0.15c/-0.15c", font="10p")
        fig.show()
        print(f"圖說：USGS，{START} 至 {endtime} UTC，M >= {MINMAG}；取樣全寬 {2*width_slider.value} km。")
        print("地形為 GMT 2 角分網格沿 A–B 中心線取樣，藍色虛線為海平面；地震是走廊內事件的投影，非斷層面本身。")
        print("地震深度沿用 USGS 目錄的參考基準，未逐筆統一垂直基準；不與地形曲線做精確絕對高程對比。")
        print(f"上下圖水平尺度相同，垂直尺度不同；VE 小於 1 表示垂直壓縮。超過 {DEPTH_MAX} km 的事件沿用色條端點色。")



map_view.on_interaction(on_map_click)
width_slider.observe(lambda change: refresh_selection(), names="value")
reset_button.on_click(reset_selection)
update_button.on_click(draw_section)
display(widgets.VBox([map_view, widgets.HBox([width_slider, update_button, reset_button]), status, output]))


## 6. 隔週繳交

作業就用 AI 做：選一段板塊交界帶，畫圖、切剖面、寫證據。

作品請把**圖＋圖說**放在一起，內容三件事：

1. **一段交界帶的地圖與至少一條 A–B 剖面**：地形當底，地震依三段深度上色、大小表規模，有比例尺、圖例與位置示意；剖面深度軸到 700 km，標 VE。可以沿用課堂範本改參數，也可以請 AI 重寫。
2. **圖說三段**：這裡看到什麼地形與地震分布；這符合哪一類交界、圖上哪些特徵是證據；哪些地方不符合或不確定，還缺什麼資料。
3. **資料註記**：來源、時間範圍、規模門檻、走廊半寬、有多少深度是預設值。

加分（自由）：再畫一段不同類型做對照；加上火山、熱點、速度剖面或震源機制當佐證；畫多條剖面看沿走向的變化。

作業只需滿足兩個條件：

1. **作品與 PyGMT 有關。**
2. **將作品上傳 GitHub，繳交 repository 連結**，並確認教師能開啟。

判斷對錯不是主要分數；圖是否完整可讀、推論是否有圖上證據、有沒有誠實寫出不確定，才是。隔週繳交即可，答案下堂課對照板塊邊界模型一起揭曉。

## 資料來源與版本

- [GMT 全球地形資料](https://docs.generic-mapping-tools.org/latest/datasets/remote-data.html)：PyGMT 載入，首次使用需連網。
- [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/)：查詢條件包含在下載網址中；單次上限 20,000 筆。
- [NOAA NCEI 火山位置 API](https://www.ngdc.noaa.gov/hazel/view/hazards/volcano/loc-search)：全新世火山，源自 Smithsonian Global Volcanism Program。
- GMT 範例檔 `@hotspots.txt`：Müller, Royer & Lawver (1993), *Geology* 21, 275–278。
- [EarthScope EMC TX2019slab](https://data.earthscope.org/app/products/portal/emc_model_viewer.html?id=EMC-TX2019slab)：Lu, Grand, Lai & Garnero (2019), *JGR Solid Earth*, doi:10.1029/2019JB017448。
- 交界帶分類：Lillie, R. J. (1999). *Whole Earth Geophysics*. Prentice Hall, ch. 2。板塊圖以 [USGS This Dynamic Planet (2006)](https://pubs.usgs.gov/imap/2800) 公有領域版本代替。
- [PyGMT 0.17 安裝文件](https://www.pygmt.org/v0.17.0/install.html)
- [原始課程參考 Notebook](https://github.com/oceanicdayi/plot_plate_boundary_pygmt/blob/main/pygmt_plot_plate_boundary.ipynb)